In [ ]:
# VAE HUMS2023 dataset

import os, json, glob, time, math, random
import numpy as np
from scipy.io import loadmat
from numpy.lib.format import open_memmap
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


DATASET_PAIRS = [
   ("./data/training_until20",
     "./data/testAIRL_from21",
     "HUMS_RF2"),
]

OUTPUT_DIR = "./vae_stndzscores_HUMSRF2_runs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


VAE_HIDDEN_DHUMS = [1024, 256, 64]   
VAE_DROPOUT     = 0.0
VAE_USE_L1_LOSS = False             
VAE_LR          = 1e-3
VAE_WEIGHT_DECAY= 1e-5
VAE_EPOCHS      = 300
VAE_BATCH       = 128
VAE_PATIENCE    = 10
VAE_VAL_SPLIT   = 0.10
VAE_MAX_SAMPLES = None
GRAD_CLIP_NORM  = 1.0


BETA_KL         = 1.0               

N_RUNS = 10
SEEDS  = list(range(N_RUNS))


PRINT_MODEL_SUMMARY_ONCE = True

EXPECTED_LEN = 4095
MAT_KEY = "xr"


USE_SCALER = False
REMOVE_DC  = False


SAVE_MODELS = True


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_folder_mat_1d(folder, key="xr", expected_len=4095):
    files = sorted(glob.glob(os.path.join(folder, "*.mat")))
    X, F = [], []
    for fp in files:
        m = loadmat(fp)
        arr = m.get(key, None)
        if arr is None:
            for v in m.values():
                if isinstance(v, np.ndarray) and v.size == expected_len:
                    arr = v; break
        if arr is None:
            continue
        x = np.asarray(arr, dtype=np.float32).reshape(-1)
        if expected_len and len(x) != expected_len:
            continue
        X.append(x); F.append(fp)
    if not X:
        raise RuntimeError(f"No usable .mat in {folder}")
    X = np.vstack(X)
    X = np.nan_to_num(X, copy=False)
    return X, np.array(F, dtype=object)

def standardize_train_test(Xtr_raw, Xte_raw, mode="train"):
    Xtr, Xte = Xtr_raw.copy(), Xte_raw.copy()
    if REMOVE_DC:
        Xtr = Xtr - Xtr.mean(axis=1, keepdHUMS=True)
        Xte = Xte - Xte.mean(axis=1, keepdHUMS=True)

    if mode.lower() == "train":
        if USE_SCALER:
            sc = StandardScaler().fit(Xtr.reshape(-1, 1))
            Xtr = sc.transform(Xtr.reshape(-1, 1)).reshape(Xtr.shape)
            Xte = sc.transform(Xte.reshape(-1, 1)).reshape(Xte.shape)
            scaler_mean = sc.mean_.astype(np.float32)
            scaler_scale = sc.scale_.astype(np.float32)
        else:
            mean_train, std_train = Xtr.flatten().mean(), Xtr.flatten().std()
            Xtr = (Xtr - mean_train) / (std_train + 1e-9)
            Xte = (Xte - mean_train) / (std_train + 1e-9)
            scaler_mean = np.array([mean_train]*Xtr.shape[1], dtype=np.float32)
            scaler_scale = np.array([std_train]*Xtr.shape[1], dtype=np.float32)

    elif mode.lower() == "per":
        if USE_SCALER:
            Xtr_scaled, Xte_scaled = [], []
            for x in Xtr:
                sc = StandardScaler()
                Xtr_scaled.append(sc.fit_transform(x.reshape(-1,1)).flatten())
            for x in Xte:
                sc = StandardScaler()
                Xte_scaled.append(sc.fit_transform(x.reshape(-1,1)).flatten())
            Xtr = np.array(Xtr_scaled)
            Xte = np.array(Xte_scaled)
            scaler_mean = np.zeros(Xtr.shape[1], dtype=np.float32)
            scaler_scale = np.ones(Xtr.shape[1], dtype=np.float32)
        else:
            Xtr = (Xtr - Xtr.mean(axis=1, keepdHUMS=True)) / (Xtr.std(axis=1, keepdHUMS=True) + 1e-9)
            Xte = (Xte - Xte.mean(axis=1, keepdHUMS=True)) / (Xte.std(axis=1, keepdHUMS=True) + 1e-9)
            scaler_mean = np.zeros(Xtr.shape[1], dtype=np.float32)
            scaler_scale = np.ones(Xtr.shape[1], dtype=np.float32)

    elif mode.lower() == "none":
        scaler_mean = np.zeros(Xtr.shape[1], dtype=np.float32)
        scaler_scale = np.ones(Xtr.shape[1], dtype=np.float32)
    else:
        raise ValueError(f"Unknown standardization mode: {mode}")

    return Xtr, Xte, scaler_mean, scaler_scale


def subsample_rows(X, max_samples, seed):
    if (max_samples is None) or (max_samples >= len(X)):
        return X
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X), size=max_samples, replace=False)
    return X[idx]


class MLPVAE(nn.Module):

    def __init__(self, in_dim: int, h_dHUMS, dropout=0.0):
        super().__init__()
        assert len(h_dHUMS) >= 1, 
        self.in_dim = in_dim
        self.latent_dim = h_dHUMS[-1]

        enc_layers = []
        d_prev = in_dim
        for d in h_dHUMS[:-1]:
            enc_layers += [nn.Linear(d_prev, d), nn.ReLU(inplace=True)]
            if dropout > 0: enc_layers += [nn.Dropout(dropout)]
            d_prev = d
        self.encoder_body = nn.Sequential(*enc_layers)
        self.fc_mu     = nn.Linear(d_prev, self.latent_dim)
        self.fc_logvar = nn.Linear(d_prev, self.latent_dim)

  
        dec_layers = []
        rev = list(reversed(h_dHUMS[:-1]))  
        d_prev = self.latent_dim
        for d in rev + [in_dim]:
            dec_layers += [nn.Linear(d_prev, d)]
            if d != in_dim:
                dec_layers += [nn.ReLU(inplace=True)]
                if dropout > 0: dec_layers += [nn.Dropout(dropout)]
            d_prev = d
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x):
        h = self.encoder_body(x)
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
      
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x, sample=True):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar) if sample else mu
        xhat = self.decode(z)
        return xhat, mu, logvar

def print_model_summary(model, in_dim=4095, device=DEVICE):
    x = torch.randn(2, in_dim).to(device)
    print(model)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total:,} | Trainable: {trainable:,}")
    model.eval()
    with torch.no_grad():
        xhat, mu, logvar = model(x, sample=True)
        z = mu
        y = model.decode(z)
    print(f"Input shape:  {tuple(x.shape)}")
    print(f"Latent shape: {tuple(mu.shape)}")
    print(f"Output shape: {tuple(y.shape)}")

@torch.no_grad()
def reconstruction_scores(model: nn.Module, X: np.ndarray, use_l1: bool) -> np.ndarray:
   
    model.eval()
    bs = 2048
    errs = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i+bs]).to(DEVICE)
        mu, logvar = model.encode(xb)
        xh = model.decode(mu)  # deterministic
        e = (xb - xh).abs().mean(dim=1) if use_l1 else ((xb - xh) ** 2).mean(dim=1)
        errs.append(e.detach().cpu().numpy().astype(np.float32))
    return np.concatenate(errs, axis=0)

def kl_divergence(mu, logvar):
   
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)

def train_one_run(Xtr: np.ndarray, seed: int, in_dim: int):
    set_seed(seed)
    Xfit = subsample_rows(Xtr, VAE_MAX_SAMPLES, seed)

    n = len(Xfit)
    n_val = max(1, int(VAE_VAL_SPLIT * n))
    idx = np.random.permutation(n)
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    Xtr_t = torch.from_numpy(Xfit[tr_idx])
    Xva_t = torch.from_numpy(Xfit[val_idx])
    Xva_dev = Xva_t.to(DEVICE, non_blocking=True)

    ds_tr = TensorDataset(Xtr_t)
    dl_tr = DataLoader(
        ds_tr,
        batch_size=VAE_BATCH,
        shuffle=True,
        drop_last=False,
        pin_memory=torch.cuda.is_available(),
        num_workers=0
    )

    model = MLPVAE(in_dim, VAE_HIDDEN_DHUMS, dropout=VAE_DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=VAE_LR, weight_decay=VAE_WEIGHT_DECAY)

    def recon_loss(xhat, x):
        if VAE_USE_L1_LOSS:
            return nn.L1Loss(reduction="none")(xhat, x).mean(dim=1)  # per-sample
        else:
            return nn.MSELoss(reduction="none")(xhat, x).mean(dim=1)

    best_val = float("inf"); best_state = None; bad = 0

    for ep in range(1, VAE_EPOCHS + 1):
        model.train()
        ep_loss = 0.0; steps = 0
        for (xb,) in dl_tr:
            xb = xb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            xhat, mu, logvar = model(xb, sample=True)
            rec = recon_loss(xhat, xb)                 
            kl  = kl_divergence(mu, logvar)            
            loss = (rec + BETA_KL * kl).mean()
            loss.backward()
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            opt.step()
            ep_loss += loss.item(); steps += 1


        model.eval()
        with torch.no_grad():
            xhat, mu, logvar = model(Xva_dev, sample=True)
            rec = recon_loss(xhat, Xva_dev)
            kl  = kl_divergence(mu, logvar)
            val_loss = (rec + BETA_KL * kl).mean().item()

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        if bad >= VAE_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    model.eval()
    return model

def ensure_channel_dir(tag, stamp):
    chan_dir = os.path.join(OUTPUT_DIR, f"{tag}_{stamp}")
    os.makedirs(chan_dir, exist_ok=True)
    return chan_dir

def write_json_safely(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def open_or_init_memmaps(chan_dir, n_runs, n_train, n_test):
    tr_path = os.path.join(chan_dir, "train_scores_runs_VAEHUMSRF2.npy")
    te_path = os.path.join(chan_dir, "test_scores_runs_VAEHUMSRF2.npy")
    if os.path.exists(tr_path) and os.path.exists(te_path):
        mm_tr = open_memmap(tr_path, mode="r+")
        mm_te = open_memmap(te_path, mode="r+")
        start = 0
        for r in range(mm_tr.shape[0]):
            if np.all(np.isnan(mm_tr[r])):
                start = r; break
        else:
            start = mm_tr.shape[0]
    else:
        mm_tr = open_memmap(tr_path, mode="w+", dtype="float32", shape=(n_runs, n_train))
        mm_te = open_memmap(te_path, mode="w+", dtype="float32", shape=(n_runs, n_test))
        mm_tr[:] = np.nan; mm_te[:] = np.nan
        mm_tr.flush(); mm_te.flush()
        start = 0
    return mm_tr, mm_te, start

stamp = "HUMSRF2"
manifest = []

for train_dir, test_dir, tag in DATASET_PAIRS:
    print(f"\n=== Channel: {tag} ===")
    Xtr_raw, Ftr = load_folder_mat_1d(train_dir, MAT_KEY, EXPECTED_LEN)
    Xte_raw, Fte = load_folder_mat_1d(test_dir,  MAT_KEY, EXPECTED_LEN)
    assert Xtr_raw.shape[1] == EXPECTED_LEN and Xte_raw.shape[1] == EXPECTED_LEN
    in_dim = EXPECTED_LEN

    Xtr, Xte, scaler_mean, scaler_scale = standardize_train_test(Xtr_raw, Xte_raw, mode="train")
    chan_dir = ensure_channel_dir(tag, stamp)

    meta = dict(
        tag=tag, stamp=stamp,
        train_dir=train_dir, test_dir=test_dir,
        n_train=int(Xtr.shape[0]), n_test=int(Xte.shape[0]),
        seeds=SEEDS,
        expected_len=EXPECTED_LEN, mat_key=MAT_KEY,
        remove_dc=REMOVE_DC, use_scaler=USE_SCALER,
        scaler_mean=scaler_mean.tolist(),
        scaler_scale=scaler_scale.tolist(),
        vae_params=dict(
            hidden_dHUMS=VAE_HIDDEN_DHUMS,
            dropout=VAE_DROPOUT,
            use_l1_loss=VAE_USE_L1_LOSS,
            lr=VAE_LR,
            weight_decay=VAE_WEIGHT_DECAY,
            epochs=VAE_EPOCHS,
            batch_size=VAE_BATCH,
            patience=VAE_PATIENCE,
            val_split=VAE_VAL_SPLIT,
            max_samples=(len(Xtr) if VAE_MAX_SAMPLES is None else min(VAE_MAX_SAMPLES, len(Xtr))),
            grad_clip_norm=GRAD_CLIP_NORM,
            beta_kl=BETA_KL,
            device=str(DEVICE),
        ),
        train_files=[str(x) for x in Ftr],
        test_files=[str(x) for x in Fte],
    )
    write_json_safely(os.path.join(chan_dir, "meta.json"), meta)

    if PRINT_MODEL_SUMMARY_ONCE:
        tmp_model = MLPVAE(in_dim, VAE_HIDDEN_DHUMS, dropout=VAE_DROPOUT).to(DEVICE)
        print_model_summary(tmp_model, in_dim=in_dim, device=DEVICE)
        del tmp_model

    mm_tr, mm_te, start_run = open_or_init_memmaps(chan_dir, N_RUNS, Xtr.shape[0], Xte.shape[0])
    print(f"memmaps: {mm_tr.shape} train, {mm_te.shape} test | resume from run {start_run}/{N_RUNS}")

    prog_path = os.path.join(chan_dir, "progress.json")
    if os.path.exists(prog_path):
        prog = json.load(open(prog_path))
        start_run = max(start_run, int(prog.get("last_completed_run", -1)) + 1)

    for r in range(start_run, N_RUNS):
        seed = SEEDS[r]
        t0 = time.time()
        model = train_one_run(Xtr, seed, in_dim)

        if SAVE_MODELS:
            ckpt_path = os.path.join(chan_dir, f"vae_run{r:02d}_best.pt")
            torch.save(model.state_dict(), ckpt_path)

        mm_tr[r] = reconstruction_scores(model, Xtr, VAE_USE_L1_LOSS)
        mm_te[r] = reconstruction_scores(model, Xte, VAE_USE_L1_LOSS)
        mm_tr.flush(); mm_te.flush()
        print(f"  [run {r+1}/{N_RUNS} | seed={seed}] wrote row {r}  ({time.time()-t0:.2f}s)")
        write_json_safely(prog_path, dict(last_completed_run=r, seeds=SEEDS))

    final_npz = os.path.join(chan_dir, f"vae_raw4095HUMS_runs_{tag}_{stamp}.npz")
    np.savez_compressed(
        final_npz,
        train_scores_runs=np.array(mm_tr),
        test_scores_runs=np.array(mm_te),
        train_files=Ftr,
        test_files=Fte,
        seeds=np.array(SEEDS, dtype=int),
        scaler_mean=scaler_mean,
        scaler_scale=scaler_scale,
        params=np.array(meta["vae_params"], dtype=object),
    )
    print(f"[saved] {final_npz}")

    manifest.append(dict(tag=tag, channel_dir=chan_dir, npz=final_npz,
                         n_train=int(Xtr.shape[0]), n_test=int(Xte.shape[0]), seeds=SEEDS))

man_path = os.path.join(OUTPUT_DIR, f"manifest_{stamp}.json")
with open(man_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"\n[done] Wrote manifest: {man_path}")


In [ ]:
# Thresholding and results

import os, re, csv, glob, json
import numpy as np
from sklearn.cluster import KMeans
from skimage.filters import threshold_otsu
from scipy.stats import genpareto
import itertools

NPZ_PATHS = sorted(glob.glob("./vae_stndzscores_HUMSRF2_runs/*/vae_raw4095HUMS_runs_*.npz"))


USE_MEAN   = True
USE_MEDIAN = False
USE_VOTE   = False 

ZSCORE_PER_RUN_BEFORE_ENSEMBLE = True
LOWER_TAIL = False

CALIBRATION_FILTER_REGEX = None
SEED = 0

OUT_DIR = "./vaeHUMS_threshold_results_runs"
os.makedirs(OUT_DIR, exist_ok=True)

EPS = 1e-12


CHRONOLOGY_FALLBACK = "natural"  


def _natural_key(name: str):
    parts = re.split(r"(\d+)", name)
    key = []
    for p in parts:
        if p.isdigit():
            key.append((1, int(p)))
        else:
            key.append((0, p))
    return tuple(key)

def _check_mode():
    modes = [USE_MEAN, USE_MEDIAN, USE_VOTE]
    if sum(bool(x) for x in modes) != 1:
        raise ValueError("Enable exactly one of USE_MEAN / USE_MEDIAN / USE_VOTE")
    if USE_MEAN:   return "mean"
    if USE_MEDIAN: return "median"
    return "vote"

def ensemble_scores(train_runs, test_runs, zscore_per_run=True):
    R = int(train_runs.shape[0])
    tr = train_runs.astype(float).copy()
    te = test_runs.astype(float).copy()

    if zscore_per_run:
        for r in range(R):
            mu = train_runs[r].mean()
            sd = train_runs[r].std() + 1e-12
            tr[r] = (train_runs[r] - mu) / sd
            te[r] = (test_runs[r]  - mu) / sd

    mode_name = _check_mode()
    if mode_name == "mean":
        return tr.mean(axis=0), te.mean(axis=0), mode_name
    if mode_name == "median":
        return np.median(tr, axis=0), np.median(te, axis=0), mode_name

    taus = np.percentile(tr, 99, axis=1)
    votes_tr = (tr > taus[:, None]).astype(int)
    votes_te = (te > taus[:, None]).astype(int)
    tr_score = votes_tr.mean(axis=0)
    te_score = votes_te.mean(axis=0)
    return tr_score, te_score, mode_name

def build_thresholds(train_scores):
    tr = np.asarray(train_scores, float)
    mu    = float(tr.mean())
    sigma = float(tr.std() + 1e-12)
    n     = int(len(tr))
    se    = sigma / max(1, np.sqrt(n))

    thresholds = {}
    thresholds['mean_plus_2std']   = mu + 2 * sigma
    thresholds['mean_plus_3std']   = mu + 3 * sigma
    thresholds['mean_minus_3std']  = mu - 3 * sigma

    thresholds['p90'] = float(np.percentile(tr, 90))
    thresholds['p95'] = float(np.percentile(tr, 95))
    thresholds['p97'] = float(np.percentile(tr, 97))
    thresholds['p98'] = float(np.percentile(tr, 98))
    thresholds['p99'] = float(np.percentile(tr, 99))

    thresholds['max']              = float(tr.max())
    thresholds['max_minus_se']     = float(tr.max() - se)
    thresholds['max_minus_2se']    = float(tr.max() - 2 * se)

    median = float(np.median(tr))
    mad    = float(np.median(np.abs(tr - median)) + 1e-12)
    thresholds['median_plus_3mad'] = median + 3 * mad

    try:
        k2 = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(tr.reshape(-1, 1))
        centers = np.sort(k2.cluster_centers_.flatten())
        thresholds['kmeans_mid'] = float(0.5 * (centers[0] + centers[1]))
    except Exception:
        pass

    try:
        thresholds['otsu'] = float(threshold_otsu(tr))
    except Exception:
        pass

    try:
        tail_n    = max(10, int(0.05 * len(tr)))
        tail_vals = np.sort(tr)[-tail_n:]
        shift     = float(tail_vals.min())
        shape, loc, scale = genpareto.fit(tail_vals - shift, floc=0)
        gpd_q     = genpareto.ppf(0.95, shape, loc, scale)
        thresholds['gpd_tail95'] = float(shift + gpd_q)
    except Exception:
        pass

    return thresholds

def filter_train_for_calibration(train_scores, train_files, regex):
    if not regex:
        return np.asarray(train_scores)
    pat = re.compile(regex)
    keep = [i for i, f in enumerate(train_files) if pat.search(os.path.basename(str(f)) or "")]
    if not keep:
        return np.asarray(train_scores)
    return np.asarray(train_scores)[keep]


def natural_order_indices(file_list):
    names = [os.path.basename(str(f)) for f in file_list]
    return np.array(sorted(range(len(names)), key=lambda i: _natural_key(names[i])), dtype=int)


def parse_day(filename):
    m = re.search(r"Day(\d+)", os.path.basename(str(filename)))
    return int(m.group(1)) if m else None

def chronology_indices(file_list):
    """
    Prefer Day(\\d+) if present; otherwise fall back per CHRONOLOGY_FALLBACK:
      - "natural": numeric-aware filename ordering
      - "alpha":   pure alphabetical
      - "mtime":   file modification time
    Returns:
      order (np.ndarray of indices), days (np.ndarray of ints or NaN)
    """
    days = [parse_day(f) for f in file_list]
    if any(d is not None for d in days):
        sub = [d if d is not None else 10**9 for d in days]
        order = np.argsort(np.array(sub))
        return order, np.array([d if d is not None else np.nan for d in days], dtype=float)

    names = np.array([os.path.basename(str(f)) for f in file_list])

    if CHRONOLOGY_FALLBACK.lower() == "natural":
        return np.array(sorted(range(len(names)), key=lambda i: _natural_key(names[i])), dtype=int), np.full(len(names), np.nan, dtype=float)

    if CHRONOLOGY_FALLBACK.lower() == "mtime":
        mtimes = np.array([os.path.getmtime(str(f)) for f in file_list], dtype=float)
        return np.argsort(mtimes), np.full(len(names), np.nan, dtype=float)

    return np.argsort(names), np.full(len(names), np.nan, dtype=float)


for npz_path in NPZ_PATHS:
    if not os.path.isfile(npz_path):
        print(f"[skip] {npz_path} not found"); continue

    d = np.load(npz_path, allow_pickle=True)
    tr_runs  = d["train_scores_runs"].astype(float)
    te_runs  = d["test_scores_runs"].astype(float)
    tr_files = d["train_files"].astype(object)
    te_files = d["test_files"].astype(object)

    tag = os.path.basename(npz_path).replace(".npz", "")
    mode_name = _check_mode()

   
    tr_ens, te_ens, _ = ensemble_scores(
        tr_runs, te_runs, zscore_per_run=ZSCORE_PER_RUN_BEFORE_ENSEMBLE
    )

   
    tr_calib = filter_train_for_calibration(tr_ens, tr_files, CALIBRATION_FILTER_REGEX)

    thresholds = build_thresholds(tr_calib)

   
    ord_idx, days = chronology_indices(te_files)
    inv_ord = np.empty_like(ord_idx)
    inv_ord[ord_idx] = np.arange(len(ord_idx))  # rank of each file in chronological order

    
    nat_idx = natural_order_indices(te_files)           
    inv_nat = np.empty_like(nat_idx); inv_nat[nat_idx] = np.arange(len(nat_idx))


    
    thr_names = sorted(list(thresholds.keys()))
    master_csv = os.path.join(OUT_DIR, f"{tag}_{mode_name}_ALL_THRESHOLDS.csv")
    with open(master_csv, "w", newline="") as f:
        w = csv.writer(f)
        header = ["filename", "score", "order_index", "day"] + [f"flagged_{t}" for t in thr_names]
        w.writerow(header)
        for i in range(len(te_files)):
            row = [os.path.basename(str(te_files[i])),
                   float(te_ens[i]),
                   int(inv_nat[i]),   
                   ""                 #no Day() used
            ]
            for t in thr_names:
                tau = thresholds[t]
                if LOWER_TAIL:
                    fl = (te_ens[i] <= (tau + EPS))     
                else:
                    fl = (te_ens[i] >= (tau - EPS))      
                row.append(int(fl))
            w.writerow(row)
    print(f"[saved] {master_csv}")

   
    counts_csv = os.path.join(OUT_DIR, f"{tag}_{mode_name}_threshold_counts.csv")
    earliest_csv = os.path.join(OUT_DIR, f"{tag}_{mode_name}_earliest_and_after.csv")
    with open(counts_csv, "w", newline="") as fc, open(earliest_csv, "w", newline="") as fe:
        wc = csv.writer(fc); we = csv.writer(fe)
        wc.writerow(["threshold_name", "tau", "n_anomalous", "n_normal"])
        we.writerow(["threshold_name", "tau",
                     "earliest_index", "earliest_day", "earliest_filename",
                     "n_anom_after_inclusive", "n_norm_after_inclusive"])

        for t in thr_names:
            tau = thresholds[t]
            if LOWER_TAIL:
                flags = (te_ens <= (tau + EPS))
            else:
                flags = (te_ens >= (tau - EPS))  

           
            thr_csv = os.path.join(OUT_DIR, f"{tag}_{mode_name}_{t}.csv")
            with open(thr_csv, "w", newline="") as ft:
                wt = csv.writer(ft)
                wt.writerow(["filename", "score", "flagged", "order_index", "day"])
                for i in range(len(te_files)):
                    wt.writerow([
                        os.path.basename(str(te_files[i])),
                        float(te_ens[i]),
                        int(flags[i]),
                        int(inv_nat[i]),   
                        ""                 # no Day()
                    ])
            print(f"[saved] {thr_csv}")

         
            n_anom = int(flags.sum())
            n_norm = int((~flags).sum())
            wc.writerow([t, f"{tau:.6f}", n_anom, n_norm])
            print(f"[{t}] τ={tau:.6f} | anomalies={n_anom} | normals={n_norm}")

            
            if n_anom == 0:
                we.writerow([t, f"{tau:.6f}", "", "", "", 0, len(flags)])
                continue

            flags_nat = flags[nat_idx]                     
            first_hit_idx = int(np.argmax(flags_nat))     
            if flags_nat[first_hit_idx] == 0:
                we.writerow([t, f"{tau:.6f}", "", "", "", 0, len(flags)])
                continue

            file_idx = nat_idx[first_hit_idx]
            earliest_file = os.path.basename(str(te_files[file_idx]))
            anom_after = int(flags_nat[first_hit_idx:].sum())
            norm_after = int((~flags_nat[first_hit_idx:]).sum())

            we.writerow([t, f"{tau:.6f}",
                         first_hit_idx,   # natural-order index
                         "",              # no day
                         earliest_file,
                         anom_after,
                         norm_after])

    print(f"[saved] {counts_csv}")
    print(f"[saved] {earliest_csv}")

   
print("\n--- Earliest anomaly by threshold (natural order) ---")
for t in thr_names:
    tau = thresholds[t]
    if LOWER_TAIL:
        flags = (te_ens <= (tau + EPS))
    else:
        flags = (te_ens >= (tau - EPS))

    n_anom = int(flags.sum())
    n_norm = int((~flags).sum())
    print(f"Threshold ({t}): {tau:.6f} | Anomalous: {n_anom} | Normal: {n_norm}")

    if n_anom == 0:
        print("  No anomalies detected.")
        continue

    # natural-order earliest
    flags_nat = flags[nat_idx]
    first_hit_idx = int(np.argmax(flags_nat))  # first True in natural order
    if flags_nat[first_hit_idx] == 0:
        print("  No anomalies detected.")
        continue

    file_idx = nat_idx[first_hit_idx]
    earliest_file = os.path.basename(str(te_files[file_idx]))
    print(f"  Earliest Anomalous File: {earliest_file}")
